In [12]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV_PATHS = ["d_exact_cifar10_inception.csv", "d_exact_cifar10_resnet18.csv", "d_exact_cifar10_vgg16.csv"]
OUT_PDFS = ["inception_cifar10.pdf", "resnet18_cifar10.pdf", "vgg_cifar10.pdf"]
TRAIN_SIZE = 45000
KL_NAME = "Gradient Swap, 1 epoch (ours)"

tol_bright = ['#4477AA', '#EE6677', '#228833', '#AA3377', '#66CCEE', '#CCBB44', '#000000']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']
markers = ["*", "o", "s", "^", "v", "D", "<", ">"]

# ---------------------------------------------------------------- journal style
plt.rcParams.update({
    # fonts: serif (Times-like), embedded as TrueType so journals accept the PDF
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # font sizes (bigger)
    "font.size": 22,
    "axes.labelsize": 28,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 20,

    # axes / ticks
    "axes.linewidth": 1.8,
    "axes.labelpad": 8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.major.pad": 6,
    "ytick.major.pad": 6,

    # legend
    "legend.frameon": True,
    "legend.fancybox": False,
    "legend.edgecolor": "0.2",
    "legend.framealpha": 0.95,
    "legend.handlelength": 2.6,
    "legend.borderpad": 0.5,
    "legend.labelspacing": 0.35,

    # output
    "savefig.dpi": 300,
})

for file_idx in range(len(CSV_PATHS)):
    # ---------------------------------------------------------------- load
    CSV_PATH = CSV_PATHS[file_idx]
    OUT_PDF = OUT_PDFS[file_idx]
    df = pd.read_csv(CSV_PATH)

    # keep K=1 for kl_faithful, keep all baselines (no K)
    is_kl = df["method"].str.startswith("kl_faithful")
    df = df[~is_kl | (df["k"] == 1)].copy()
    df.loc[df["method"].str.startswith("kl_faithful"), "method"] = KL_NAME
    df.loc[df["method"].str.startswith("facility_location"), "method"] = "Facility Location"
    df.loc[df["method"].str.startswith("graphcut"), "method"] = "GraphCut"
    df.loc[df["method"].str.startswith("herding"), "method"] = "Herding"
    df.loc[df["method"].str.startswith("moderate"), "method"] = "Moderate Coreset"
    df.loc[df["method"].str.startswith("moso"), "method"] = "MoSo"
    df.loc[df["method"].str.startswith("random"), "method"] = "Random"
    df.loc[df["method"].str.startswith("el2n"), "method"] = "EL2N"
    df["frac"] = df["budget"] / TRAIN_SIZE
    df["acc"] = 100.0 - df["error"]          # 'error' is test error in %

    # drop EL2N's first (smallest-fraction) point
    el2n_first = df.loc[df["method"] == "EL2N", "frac"].min()
    df = df[~((df["method"] == "EL2N") & (df["frac"] == el2n_first))]

    # ---------------------------------------------------------------- aggregate
    agg = (df.groupby(["method", "frac"])["acc"]
            .agg(mean="mean", std="std", n="count")
            .reset_index())
    agg["std"] = agg["std"].fillna(0.0)
    print(agg.to_string(index=False))        # n = number of seeds per point

    # ---------------------------------------------------------------- style
    # ours gets slot 0; baselines follow, with EL2N last (it is mostly off-plot)
    others = [m for m in sorted(agg["method"].unique()) if m not in (KL_NAME, "EL2N")]
    if "EL2N" in set(agg["method"]):
        others.append("EL2N")
    order = [KL_NAME] + others

    def get_style(idx):
        # 8 curves but 7 colors: on wrap-around, shift the linestyle so that
        # (color, linestyle) pairs stay unique
        return dict(color=tol_bright[idx % 7],
                    ls=linestyles[(idx + idx // 7) % 7],
                    marker=markers[idx % len(markers)])

    fracs = sorted(agg["frac"].unique())

    # ---------------------------------------------------------------- plot
    fig, ax = plt.subplots(figsize=(10, 7.5))

    for idx, m in enumerate(order):
        d = agg[agg["method"] == m].sort_values("frac")
        s = get_style(idx)
        is_ours = (m == KL_NAME)
        ax.plot(d["frac"], d["mean"], color=s["color"], linestyle=s["ls"],
                marker=s["marker"], lw=3.4 if is_ours else 2.6,
                ms=18 if is_ours else 10, label=m,
                markeredgecolor=s["color"], zorder=10 if is_ours else 3)
        ax.fill_between(d["frac"], d["mean"] - d["std"], d["mean"] + d["std"],
                        color=s["color"], alpha=0.15, lw=0, zorder=2)

    # y-limits set from everything except EL2N, so EL2N runs off the bottom
    ref = agg[agg["method"] != "EL2N"]
    lo = (ref["mean"] - ref["std"]).min()
    hi = (ref["mean"] + ref["std"]).max()
    pad = 0.06 * (hi - lo)
    ax.set_ylim(lo - pad, hi + pad)

    ax.set_xticks(fracs)
    ax.set_xlim(min(fracs) - 0.04, max(fracs) + 0.04)
    ax.set_xlabel("Fraction of the training set")
    ax.set_ylabel("Test accuracy (%)")
    ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right")

    fig.tight_layout()
    fig.savefig(OUT_PDF, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {OUT_PDF}")

                       method  frac      mean      std  n
                         EL2N   0.2 68.073667 2.155596  5
                         EL2N   0.4 90.445667 0.398111  5
                         EL2N   0.5 91.654000 0.269559  5
            Facility Location   0.1 71.482667 1.580256  5
            Facility Location   0.2 82.887667 0.526634  5
            Facility Location   0.4 89.533000 0.132934  5
            Facility Location   0.5 90.829000 0.413262  5
Gradient Swap, 1 epoch (ours)   0.1 75.343667 1.107763  5
Gradient Swap, 1 epoch (ours)   0.2 83.545667 0.311200  5
Gradient Swap, 1 epoch (ours)   0.4 88.427667 0.207232  5
Gradient Swap, 1 epoch (ours)   0.5 89.457333 0.237787  5
                     GraphCut   0.1 75.966667 0.526990  5
                     GraphCut   0.2 82.563000 0.206423  5
                     GraphCut   0.4 86.859333 0.339179  5
                     GraphCut   0.5 88.324667 0.109439  5
                      Herding   0.1 73.924333 0.725726  5
              